# Generating fields from user-supplied standard normal deviates

`fieldgen2d_sva()` and `fieldgen3d_sva()` generate stochastic fields by convolving an averaging function (whose shape can vary from cell to cell) with a field of independent standard normal deviates, which are drawn internally from the library's random number generator.

`fieldgen2d_sva_z()` and `fieldgen3d_sva_z()` do exactly the same convolution, but **you supply the deviates** (`z`). This means:

 - the "white noise" behind a realisation is visible and under your control,
 - realisations are reproducible without seeding the Fortran generator,
 - you can hold `z` fixed while changing the variogram (common random numbers),
 - `z` can come from anywhere: `numpy`, an existing ensemble, or a parameter-estimation process that treats `z` as the (uncorrelated, unit-variance) adjustable parameters.

This notebook demonstrates each of these.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.append("..")
from pypestutils.pestutilslib import PestUtilsLib

lib = PestUtilsLib(logger_level=0)

## Setup

An 80 x 120 grid of unit cells. The field mean is `1.0` and variance `0.1` (think log10 hydraulic conductivity). We use an exponential averaging function (`avetype=2`).

In [ ]:
nrow, ncol = 80, 120
x, y = np.meshgrid(np.arange(ncol) + 0.5, np.arange(nrow)[::-1] + 0.5)
ec, nc = x.ravel(), y.ravel()
nnode = ec.size
area = 1.0
active = 1
mean = 1.0
var = 0.1
transtype = 0  # natural (not log) transform of the field
avetype = 2  # exponential
power = 0.0


def gen(z, aa=10.0, anis=1.0, bearing=0.0, mean=mean, var=var):
    """Thin helper around fieldgen2d_sva_z; returns (nreal, nrow, ncol)."""
    f = lib.fieldgen2d_sva_z(
        ec,
        nc,
        area,
        active,
        mean,
        var,
        aa,
        anis,
        bearing,
        transtype,
        avetype,
        power,
        z,
    )
    return f.T.reshape(-1, nrow, ncol)


def show(arrs, titles, vmin=0.0, vmax=2.0, cmap="viridis"):
    fig, axes = plt.subplots(1, len(arrs), figsize=(3.6 * len(arrs), 2.8))
    axes = np.atleast_1d(axes)
    for ax, a, t in zip(axes, arrs, titles):
        cb = ax.imshow(a, vmin=vmin, vmax=vmax, cmap=cmap)
        ax.set_title(t, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(cb, ax=axes.tolist(), shrink=0.8)
    return fig

## 1. `z` is the white noise behind a realisation

Draw `z` with numpy (no call to `initialize_randgen()` is needed), and look at the noise next to the field it produces.

In [ ]:
rng = np.random.default_rng(42)
z = rng.standard_normal(nnode)  # one realisation; shape (nnode,) or (nnode, nreal)

field = gen(z)[0]
_ = show(
    [mean + np.sqrt(var) * z.reshape(nrow, ncol), field],
    ["white noise: mean + sqrt(var) * z", "fieldgen2d_sva_z(z), aa=10"],
)

The same `z` always produces the same field, and `z = 0` produces the mean everywhere:

In [ ]:
assert np.array_equal(gen(z)[0], field)
assert np.allclose(gen(np.zeros(nnode))[0], mean)
print("deterministic and centred on the mean")

## 2. Common random numbers: same `z`, different variograms

Holding `z` fixed isolates the effect of the variogram from sampling noise, which is useful for building intuition and for sensitivity analysis. (This is what the `understanding_variograms_and_realizations` notebook achieves by re-seeding; here it is explicit.)

In [ ]:
aas = [3.0, 10.0, 30.0]
_ = show([gen(z, aa=a)[0] for a in aas], [f"aa={a:g}" for a in aas])

bearings = [0.0, 45.0, 90.0]
_ = show(
    [gen(z, aa=20.0, anis=5.0, bearing=b)[0] for b in bearings],
    [f"anis=5, bearing={b:g}" for b in bearings],
)

## 3. Impulse response: seeing the spatially varying averaging function

Because the field is a *linear* function of `z` (for `transtype=0`), setting `z` to zero everywhere except at a few nodes shows the averaging function itself. Here the bearing rotates from 0° on the left of the grid to 90° on the right, and the anisotropy increases from bottom to top. Each "blob" is the footprint of one non-zero deviate.

(`z` must be supplied for every node, active or not; it is the convolution that spreads each deviate over its neighbours.)

In [ ]:
bearing_sv = 90.0 * ec / ec.max()  # 0 -> 90 degrees, left to right
anis_sv = 1.0 + 5.0 * nc / nc.max()  # 1 -> 6, bottom to top

zi = np.zeros((nrow, ncol))
for r in (15, 40, 65):
    for c in (15, 45, 75, 105):
        zi[r, c] = 1.0

imp = gen(zi.ravel(), aa=12.0, anis=anis_sv, bearing=bearing_sv, mean=0.0, var=1.0)[0]
_ = show(
    [zi, imp],
    ["impulses in z", "response (spatially varying anis/bearing)"],
    vmin=0.0,
    vmax=imp.max(),
    cmap="magma",
)

## 4. Linearity: fields can be built up from pieces of `z`

For `transtype=0`, `f(a*z1 + b*z2) - mean = a*(f(z1) - mean) + b*(f(z2) - mean)`. For `transtype=1` the same holds in log10 space: `log10(f/mean)` is linear in `z`.

In [ ]:
z1, z2 = rng.standard_normal((2, nnode))
f1, f2, f12 = (gen(zz)[0] - mean for zz in (z1, z2, 0.7 * z1 - 1.3 * z2))
print("max linearity error:", np.abs(f12 - (0.7 * f1 - 1.3 * f2)).max())

## 5. Gradual deformation: smoothly morphing between realisations

If `z1` and `z2` are independent standard normal fields, then `z(t) = cos(t)·z1 + sin(t)·z2` is also standard normal for every `t`. So every intermediate field is an equally valid realisation of the same geostatistical model, and the sequence morphs smoothly from one realisation to the other. This is the basis of the *gradual deformation method* used in history matching. It is only possible when the deviates are under your control.

In [ ]:
ts = np.linspace(0, np.pi / 2, 5)
zt = np.column_stack([np.cos(t) * z1 + np.sin(t) * z2 for t in ts])
morph = gen(zt, aa=15.0, anis=3.0, bearing=30.0)
_ = show(list(morph), [f"t = {t / (np.pi / 2):.2f} x pi/2" for t in ts])

## 6. Ensembles and statistics

Pass a 2D `z` with shape `(nnode, nreal)` to generate many realisations in one call. Because the kernel normalises each node's weights, the variance of the generated field reproduces `var` (to within sampling error), wherever the averaging function changes.

Ensembles are costly (work grows with nodes² × realisations), so we use a coarser 40 x 60 grid here.

In [ ]:
nrow_c, ncol_c = 40, 60
xc, yc = np.meshgrid(np.arange(ncol_c) * 2.0 + 1.0, np.arange(nrow_c)[::-1] * 2.0 + 1.0)
ecc, ncc = xc.ravel(), yc.ravel()
bearing_c = 90.0 * ecc / ecc.max()
anis_c = 1.0 + 5.0 * ncc / ncc.max()

nreal = 300
Z = rng.standard_normal((ecc.size, nreal))
ens = lib.fieldgen2d_sva_z(
    ecc,
    ncc,
    4.0,
    active,
    mean,
    var,
    12.0,
    anis_c,
    bearing_c,
    transtype,
    avetype,
    power,
    Z,
)
print("ensemble shape:", ens.shape)
print(f"node-wise mean: {ens.mean(axis=1).mean():.3f} (target {mean})")
print(f"node-wise var : {ens.var(axis=1).mean():.3f} (target {var})")

fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
for ax, a, t in zip(
    axes, [ens.mean(axis=1), ens.var(axis=1)], ["ensemble mean", "ensemble variance"]
):
    cb = ax.imshow(a.reshape(nrow_c, ncol_c), cmap="viridis")
    ax.set_title(t, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(cb, ax=ax, shrink=0.8)

## 7. Relationship to `fieldgen2d_sva()`

`fieldgen2d_sva()` is now a thin wrapper: it draws `z` from the seeded Fortran generator and calls the same kernel as `fieldgen2d_sva_z()`. The two give statistically identical ensembles; they differ only in where `z` comes from.

In [ ]:
lib.initialize_randgen(1234)
ens_internal = lib.fieldgen2d_sva(
    ecc,
    ncc,
    4.0,
    active,
    mean,
    var,
    12.0,
    anis_c,
    bearing_c,
    transtype,
    avetype,
    power,
    nreal,
)
print(f"fieldgen2d_sva   node-wise var: {ens_internal.var(axis=1).mean():.4f}")
print(f"fieldgen2d_sva_z node-wise var: {ens.var(axis=1).mean():.4f}")

## 3D

`fieldgen3d_sva_z()` works the same way: it takes the arguments of `fieldgen3d_sva()`, with `z` in place of `nreal`.

In [ ]:
nlay = 4
x3 = np.tile(ecc, nlay)
y3 = np.tile(ncc, nlay)
z3 = np.repeat(np.arange(nlay)[::-1] * 5.0 + 2.5, ecc.size)
Z3 = rng.standard_normal((x3.size, 1))
f3 = lib.fieldgen3d_sva_z(
    x3,
    y3,
    z3,
    4.0,
    5.0,
    1,
    mean,
    var,
    20.0,
    8.0,
    6.0,  # ahmax, ahmin, avert
    30.0,
    0.0,
    0.0,  # bearing, dip, rake
    transtype,
    avetype,
    power,
    Z3,
)
fig, axes = plt.subplots(1, nlay, figsize=(3.2 * nlay, 2.4))
for k, ax in enumerate(axes):
    cb = ax.imshow(f3[:, 0].reshape(nlay, nrow_c, ncol_c)[k], vmin=0.0, vmax=2.0)
    ax.set_title(f"layer {k + 1}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(cb, ax=axes.tolist(), shrink=0.8)

In [ ]:
lib.free_all_memory()